In [0]:
# Cell 1: Imports and Loading Bronze Data
from pyspark.sql.functions import col, to_date

# Read directly from the persistent Bronze Delta tables we just created
bronze_policies = spark.table("workspace.default.bronze_insurance_policies")
bronze_claims   = spark.table("workspace.default.bronze_insurance_claims")
bronze_web_log  = spark.table("workspace.default.bronze_web_events")

print("Bronze data successfully pulled into the Silver session.")

Bronze data successfully pulled into the Silver session.


In [0]:
    # Cell 2: Cleansing and Transforming Data

# 1. Clean Policies Table
silver_policies_df = bronze_policies \
    .withColumn("premium_amount", col("premium_amount").cast("double")) \
    .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["policy_id"])

# 2. Clean Claims Table
silver_claims_df = bronze_claims \
    .withColumn("claim_amount", col("claim_amount").cast("double")) \
    .withColumn("incident_date", to_date(col("incident_date"), "yyyy-MM-dd")) \
    .filter(col("claim_id").isNotNull()) \
    .dropDuplicates(["claim_id"])

# 3. Clean Web Funnel Logs (Converting ISO timestamps into true database timestamp types)
silver_web_df = bronze_web_log \
    .withColumn("timestamp", col("timestamp").cast("timestamp")) \
    .filter(col("session_id").isNotNull())

print("All transformations applied successfully.")

All transformations applied successfully.


In [0]:
# Cell 3: Writing Tables out to the Silver Layer Catalog

silver_policies_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_insurance_policies")

silver_claims_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_insurance_claims")

silver_web_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_web_events")

print("Success! All 3 tables have been promoted to the Silver Layer.")

Success! All 3 tables have been promoted to the Silver Layer.


In [0]:
%sql
select policy_id, policy_type, premium_amount, signup_date
from workspace.default.silver_insurance_policies
limit 10;

policy_id,policy_type,premium_amount,signup_date
POL_5027,Auto,2993.31,2026-03-07
POL_5067,Auto,1960.44,2026-01-08
POL_5088,Home,903.95,2026-03-12
POL_5009,Auto,2893.03,2026-01-14
POL_5017,Home,906.64,2026-01-27
POL_5020,Home,1448.64,2026-03-29
POL_5034,Home,508.11,2026-03-06
POL_5055,Health,2960.41,2026-01-08
POL_5056,Home,2320.53,2026-02-01
POL_5057,Auto,975.52,2026-01-18
